# Module 3.1: RULER Hands-On

## Learning Objectives

By completing this notebook, you will:
1. Understand why relative scoring beats absolute scoring for LLMs
2. Implement an LLM-as-a-judge scoring function
3. Build a hybrid reward combining programmatic + RULER scores
4. Validate reward functions on sample trajectories

## Estimated Time: 15 minutes

---

## 1. The Reward Function Problem

The hardest part of RL: defining what "good" means.

- **Too simple** (binary correct/incorrect): misses quality differences
- **Too complex** (custom scoring): brittle, domain-specific, expensive to maintain
- **Manual labels**: doesn't scale

RULER solves this by using an LLM to judge trajectory quality **relatively**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The problem with absolute scoring
# An LLM asked to "rate this response 0-10" is inconsistent

np.random.seed(42)

# Simulate: same response scored 5 times by an absolute judge
absolute_scores_response_a = np.random.normal(6.5, 1.5, 50).clip(0, 10)
absolute_scores_response_b = np.random.normal(7.0, 1.5, 50).clip(0, 10)

# Simulate: same pair compared 50 times by a relative judge
# "Which is better, A or B?" is much more consistent
relative_judgments = np.random.binomial(1, 0.65, 50)  # B wins 65% of the time

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Absolute scoring: noisy, overlapping distributions
axes[0].hist(absolute_scores_response_a, alpha=0.6, bins=15, label="Response A")
axes[0].hist(absolute_scores_response_b, alpha=0.6, bins=15, label="Response B")
axes[0].set_title("Absolute Scoring: Noisy & Inconsistent")
axes[0].set_xlabel("Score (0-10)")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].axvline(absolute_scores_response_a.mean(), color="C0", linestyle="--")
axes[0].axvline(absolute_scores_response_b.mean(), color="C1", linestyle="--")

# Relative scoring: clear signal
a_wins = (1 - relative_judgments).sum()
b_wins = relative_judgments.sum()
axes[1].bar(["A wins", "B wins"], [a_wins, b_wins], color=["C0", "C1"])
axes[1].set_title("Relative Scoring: Clear & Consistent")
axes[1].set_ylabel("Count (out of 50)")

plt.tight_layout()
plt.show()

print(f"Absolute scoring overlap makes it hard to distinguish A from B.")
print(f"Relative scoring: B is better than A {b_wins}/50 times = clear signal.")

## 2. RULER's Two Key Insights

1. **Relative scoring > absolute scoring** for LLMs. "Which of these 4 is best?" is more reliable than "Rate this 0-10."

2. **GRPO only needs relative scores.** Since GRPO normalizes rewards within each group, absolute values don't matter — only the ordering.

In [ ]:
# GRPO doesn't care about absolute values — only relative ordering
# These two sets of rewards produce IDENTICAL training signals

def compute_advantages(rewards):
    r = np.array(rewards, dtype=np.float64)
    return ((r - r.mean()) / (r.std() + 1e-8)).tolist()

# Scenario 1: Low absolute scores
rewards_low = [0.1, 0.3, 0.5, 0.7]
advantages_low = compute_advantages(rewards_low)

# Scenario 2: High absolute scores (same ordering)
rewards_high = [0.4, 0.6, 0.8, 1.0]
advantages_high = compute_advantages(rewards_high)

print("Low scores:  ", [f"{r:.1f}" for r in rewards_low])
print("Advantages:  ", [f"{a:+.2f}" for a in advantages_low])
print()
print("High scores: ", [f"{r:.1f}" for r in rewards_high])
print("Advantages:  ", [f"{a:+.2f}" for a in advantages_high])
print()
print("Same advantages! GRPO only cares about relative ordering.")

## 3. Implementing RULER Scoring

RULER sends N trajectories to an LLM judge and gets back scores.

In [ ]:
RULER_SYSTEM_PROMPT = """You are an expert judge evaluating AI agent performance.

You will receive a task description and multiple agent trajectories.
Score each trajectory from 0.0 to 1.0 based on:
- Correctness: Did the agent produce the right answer?
- Efficiency: Did it use minimal tool calls?
- Robustness: Did it explore the schema before querying?

Output ONLY a JSON array of scores. Example: [0.9, 0.3, 0.7, 0.1]"""


def build_ruler_prompt(task: str, trajectories: list[str]) -> str:
    """Build the judge prompt for RULER scoring."""
    parts = [f"Task: {task}\n"]
    for i, traj in enumerate(trajectories):
        parts.append(f"--- Trajectory {i+1} ---")
        parts.append(traj)
        parts.append("")
    parts.append(f"Score each trajectory 0.0-1.0. Return ONLY a JSON array of {len(trajectories)} scores.")
    return "\n".join(parts)


# Build a sample prompt
task = "Find all employees earning more than $150,000"
trajectories = [
    """[user] Find all employees earning more than $150,000
[assistant] -> list_tables()
[tool] departments, employees, projects
[assistant] -> describe_table('employees')
[tool] id, name, department_id, role, salary, hire_date
[assistant] -> run_query('SELECT name, salary FROM employees WHERE salary > 150000 ORDER BY salary DESC')
[tool] Bob Martinez|195000, Jack Wilson|175000, David Kim|170000, Alice Chen|165000, Iris Nakamura|160000, Eva Patel|155000
[assistant] There are 6 employees earning more than $150,000.""",
    
    """[user] Find all employees earning more than $150,000
[assistant] -> run_query('SELECT * FROM employees WHERE salary > 150000')
[tool] (6 rows returned)
[assistant] There are 6 employees earning over $150K.""",
    
    """[user] Find all employees earning more than $150,000
[assistant] -> run_query('SELECT * FROM emp WHERE pay > 150000')
[tool] SQL Error: no such table: emp
[assistant] I believe there are about 5 employees earning more than $150,000.""",
]

prompt = build_ruler_prompt(task, trajectories)
print(prompt[:500])
print("...")

In [ ]:
# Simulated RULER output (in production, this comes from the judge LLM)
# The judge correctly identifies:
# - Trajectory 1: Best (explored schema, correct query, detailed answer)
# - Trajectory 2: OK (correct result but no schema exploration)
# - Trajectory 3: Bad (wrong table name, hallucinated answer)

ruler_scores = [0.95, 0.6, 0.1]

advantages = compute_advantages(ruler_scores)

labels = ["Explored + correct", "Direct query + correct", "Error + hallucinated"]
colors = ["green" if a > 0 else "red" for a in advantages]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(labels, ruler_scores, color=colors, alpha=0.7)
axes[0].set_xlabel("RULER Score")
axes[0].set_title("Raw RULER Scores")
axes[0].set_xlim(0, 1)

axes[1].barh(labels, advantages, color=colors, alpha=0.7)
axes[1].set_xlabel("GRPO Advantage")
axes[1].set_title("GRPO Training Signal")
axes[1].axvline(0, color="black", linewidth=0.5)

plt.tight_layout()
plt.show()

## 4. Hybrid Rewards

Combine programmatic checks (fast, deterministic) with RULER (nuanced, subjective).

In [ ]:
import sqlite3


def programmatic_sql_reward(response: str, db_path: str = ":memory:") -> float:
    """
    Score based on SQL validity and result correctness.
    
    Scoring:
    - 0.0: No SQL found in response
    - 0.2: SQL found but has errors
    - 0.5: Valid SQL, returns results
    - 1.0: Valid SQL, results match expected
    """
    import re
    
    # Extract SQL from response
    match = re.search(r"SELECT.*?;", response, re.DOTALL | re.IGNORECASE)
    if not match:
        return 0.0
    
    sql = match.group(0)
    
    # Try to execute
    try:
        conn = sqlite3.connect(db_path)
        rows = conn.execute(sql).fetchall()
        conn.close()
        return 0.5 if rows else 0.3
    except sqlite3.Error:
        return 0.2


def hybrid_reward(
    response: str,
    ruler_score: float,
    programmatic_weight: float = 0.4,
    ruler_weight: float = 0.6,
) -> float:
    """Combine programmatic and RULER scores."""
    prog_score = programmatic_sql_reward(response)
    return programmatic_weight * prog_score + ruler_weight * ruler_score


# Example
responses = [
    "The query is: SELECT name, salary FROM employees WHERE salary > 150000;",
    "I think the answer is about 5 employees.",
    "SELECT * FORM employees;"  # syntax error
]
ruler_scores = [0.9, 0.2, 0.1]

print("Hybrid Reward Scoring")
print("=" * 50)
for resp, rs in zip(responses, ruler_scores):
    prog = programmatic_sql_reward(resp)
    hyb = hybrid_reward(resp, rs)
    print(f"  Programmatic: {prog:.2f} | RULER: {rs:.2f} | Hybrid: {hyb:.2f}")
    print(f"  Response: {resp[:60]}...")
    print()

## Exercise: Design a Reward Function

**Task:** Implement a reward function for a code review agent.

The agent reviews Python code and should:
1. Identify actual bugs (not just style issues)
2. Suggest specific fixes
3. Explain why the fix matters

Implement `code_review_reward` that checks for these criteria.

In [ ]:
# YOUR CODE HERE
# ---------------

def code_review_reward(review: str) -> float:
    """
    Score a code review response.
    
    Scoring criteria:
    - Has 'bug' or 'issue' or 'error' mentioned: +0.3
    - Has a code fix (contains ```) : +0.4
    - Has explanation of why (contains 'because' or 'since' or 'reason'): +0.3
    
    Returns float between 0.0 and 1.0
    """
    # TODO: Implement this
    pass

exercise_answer = code_review_reward

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise():
    # Good review: has all three components
    good_review = """There's a bug in the loop. Here's the fix:
    ```python
    for i in range(len(items)):
    ```
    This is important because the original code causes an IndexError."""
    
    score = exercise_answer(good_review)
    assert score is not None, "Function should return a value"
    assert isinstance(score, float), "Should return a float"
    assert score >= 0.8, f"Good review should score >= 0.8, got {score}"
    
    # Bad review: no substance
    bad_review = "Looks fine to me."
    score = exercise_answer(bad_review)
    assert score <= 0.3, f"Bad review should score <= 0.3, got {score}"
    
    # Partial review: identifies issue but no fix
    partial = "There's a potential error in the authentication logic."
    score = exercise_answer(partial)
    assert 0.2 <= score <= 0.5, f"Partial review should be 0.2-0.5, got {score}"
    
    print("All tests passed!")

test_exercise()

## Summary

**Key Takeaways:**
1. Relative scoring ("which is best?") is more reliable than absolute ("rate 0-10")
2. GRPO only needs relative ordering — absolute values don't matter
3. RULER uses an LLM judge to score trajectories without labeled data
4. Hybrid rewards combine programmatic checks with RULER for best results

**Next:** Module 04 covers building MCP tool servers for agent environments.